In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [16]:
train_data = pd.read_csv("../data/processed/train.csv")
test_data = pd.read_csv("../data/processed/test.csv")

In [17]:
print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

Train shape: (800167, 4)
Test shape: (200042, 4)


In [18]:
print("Số user trong Train:", train_data["user_id"].nunique())
print("Số movie trong Train:", train_data["movie_id"].nunique())

print(
    "User chỉ có trong Test:",
    len(set(test_data["user_id"]) - set(train_data["user_id"]))
)

print(
    "Movie chỉ có trong Test:",
    len(set(test_data["movie_id"]) - set(train_data["movie_id"]))
)

Số user trong Train: 6040
Số movie trong Train: 3683
User chỉ có trong Test: 0
Movie chỉ có trong Test: 23


In [19]:
required_cols = {"user_id", "movie_id", "rating"}

assert required_cols.issubset(train_data.columns)
assert required_cols.issubset(test_data.columns)

# Ánh xạ ID sang chỉ số liên tục
user_ids = np.sort(train_data["user_id"].unique())
movie_ids = np.sort(train_data["movie_id"].unique())

user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
movie_to_idx = {mid: idx for idx, mid in enumerate(movie_ids)}

idx_to_user = {idx: uid for uid, idx in user_to_idx.items()}
idx_to_movie = {idx: mid for mid, idx in movie_to_idx.items()}

# Tạo dữ liệu huấn luyện dạng số
train_users = train_data["user_id"].map(user_to_idx).to_numpy()
train_movies = train_data["movie_id"].map(movie_to_idx).to_numpy()
train_ratings = train_data["rating"].to_numpy(dtype=np.float32)

# Chỉ giữ những tương tác Test có user và movie đã xuất hiện trong Train
test_eval = test_data[
    test_data["user_id"].isin(user_to_idx)
    & test_data["movie_id"].isin(movie_to_idx)
].copy()

print("Số user:", len(user_ids))
print("Số movie:", len(movie_ids))
print("Số tương tác Train:", len(train_ratings))
print("Số tương tác Test có thể đánh giá:", len(test_eval))
print("Số tương tác Test bị loại do movie chưa có trong Train:",
      len(test_data) - len(test_eval))

Số user: 6040
Số movie: 3683
Số tương tác Train: 800167
Số tương tác Test có thể đánh giá: 200016
Số tương tác Test bị loại do movie chưa có trong Train: 26


In [20]:
# Cấu hình mô hình
N_FACTORS = 50
LEARNING_RATE = 0.005
REGULARIZATION = 0.02
N_EPOCHS = 15
RANDOM_STATE = 42

# Số lượng user và movie
n_users = len(user_ids)
n_movies = len(movie_ids)

# Khởi tạo tham số
rng = np.random.default_rng(RANDOM_STATE)

global_mean = float(np.mean(train_ratings))

user_bias = np.zeros(n_users, dtype=np.float32)
movie_bias = np.zeros(n_movies, dtype=np.float32)

user_factors = rng.normal(
    loc=0.0,
    scale=0.05,
    size=(n_users, N_FACTORS)
).astype(np.float32)

movie_factors = rng.normal(
    loc=0.0,
    scale=0.05,
    size=(n_movies, N_FACTORS)
).astype(np.float32)

print("Global mean:", round(global_mean, 4))
print("User factors:", user_factors.shape)
print("Movie factors:", movie_factors.shape)
print("Số tham số latent factors:",
      user_factors.size + movie_factors.size)

Global mean: 3.5813
User factors: (6040, 50)
Movie factors: (3683, 50)
Số tham số latent factors: 486150


In [21]:

def train_mf_sgd(
    train_users,
    train_movies,
    train_ratings,
    n_users,
    n_movies,
    n_factors=20,
    learning_rate=0.005,
    reg=0.02,
    n_epochs=15,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    global_mean = float(np.mean(train_ratings))

    user_bias = np.zeros(n_users, dtype=np.float32)
    movie_bias = np.zeros(n_movies, dtype=np.float32)

    user_factors = rng.normal(
        0, 0.05, (n_users, n_factors)
    ).astype(np.float32)

    movie_factors = rng.normal(
        0, 0.05, (n_movies, n_factors)
    ).astype(np.float32)

    n = len(train_ratings)
    history = []

    for epoch in range(n_epochs):
        # Xáo trộn thứ tự tương tác mỗi epoch
        order = rng.permutation(n)

        squared_error = 0.0

        for j in order:
            u = train_users[j]
            i = train_movies[j]
            r = train_ratings[j]

            pu = user_factors[u].copy()
            qi = movie_factors[i].copy()

            pred = (
                global_mean
                + user_bias[u]
                + movie_bias[i]
                + np.dot(pu, qi)
            )

            err = r - pred
            squared_error += err * err

            # Cập nhật bias
            user_bias[u] += learning_rate * (
                err - reg * user_bias[u]
            )
            movie_bias[i] += learning_rate * (
                err - reg * movie_bias[i]
            )

            # Cập nhật latent factors
            user_factors[u] += learning_rate * (
                err * qi - reg * pu
            )
            movie_factors[i] += learning_rate * (
                err * pu - reg * qi
            )

        train_rmse = np.sqrt(squared_error / n)
        history.append(train_rmse)

        print(
            f"Epoch {epoch + 1:02d}/{n_epochs} "
            f"| Train RMSE: {train_rmse:.4f}"
        )

    model = {
        "global_mean": global_mean,
        "user_bias": user_bias,
        "movie_bias": movie_bias,
        "user_factors": user_factors,
        "movie_factors": movie_factors,
        "history": history,
    }

    return model


In [22]:

mf_model = train_mf_sgd(
    train_users=train_users,
    train_movies=train_movies,
    train_ratings=train_ratings,
    n_users=n_users,
    n_movies=n_movies,
    n_factors=N_FACTORS,
    learning_rate=LEARNING_RATE,
    reg=REGULARIZATION,
    n_epochs=N_EPOCHS,
    random_state=RANDOM_STATE
)


Epoch 01/15 | Train RMSE: 0.9817
Epoch 02/15 | Train RMSE: 0.9243
Epoch 03/15 | Train RMSE: 0.9117
Epoch 04/15 | Train RMSE: 0.9057
Epoch 05/15 | Train RMSE: 0.9018
Epoch 06/15 | Train RMSE: 0.8989
Epoch 07/15 | Train RMSE: 0.8960
Epoch 08/15 | Train RMSE: 0.8926
Epoch 09/15 | Train RMSE: 0.8882
Epoch 10/15 | Train RMSE: 0.8824
Epoch 11/15 | Train RMSE: 0.8751
Epoch 12/15 | Train RMSE: 0.8668
Epoch 13/15 | Train RMSE: 0.8579
Epoch 14/15 | Train RMSE: 0.8489
Epoch 15/15 | Train RMSE: 0.8396


In [23]:
def recommend_top_n(user_id, model, n=10):
    if user_id not in user_to_idx:
        return pd.DataFrame(columns=["movie_id", "score"])

    u = user_to_idx[user_id]

    # Tính điểm cho tất cả phim trong tập Train
    scores = (
        model["global_mean"]
        + model["user_bias"][u]
        + model["movie_bias"]
        + model["movie_factors"] @ model["user_factors"][u]
    )

    # Loại phim người dùng đã tương tác trong Train
    seen_movies = set(
        train_data.loc[
            train_data["user_id"] == user_id, "movie_id"
        ]
    )

    candidates = pd.DataFrame({
        "movie_id": movie_ids,
        "score": scores
    })

    candidates = candidates[
        ~candidates["movie_id"].isin(seen_movies)
    ]

    return candidates.nlargest(n, "score").reset_index(drop=True)

In [24]:
sample_user = int(train_data["user_id"].iloc[0])

recommendations = recommend_top_n(
    user_id=sample_user,
    model=mf_model,
    n=10
)

print("User ID:", sample_user)
display(recommendations)

User ID: 2507


,movie_id,score
0,750,5.188526
1,2019,5.035566
2,1219,5.009897
3,1208,4.932462
4,2395,4.931038
5,1221,4.930125
6,3089,4.926218
7,3134,4.885502
8,162,4.881215
9,1207,4.863832


In [25]:
RELEVANCE_THRESHOLD = 4
TOP_N = 10

relevant_test = test_eval[
    test_eval["rating"] >= RELEVANCE_THRESHOLD
].copy()

# Tập hợp các phim liên quan của từng người dùng
relevant_items_by_user = (
    relevant_test.groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)

# Chỉ đánh giá những user có ít nhất một phim liên quan trong Test
eval_users = sorted(relevant_items_by_user.keys())

print("Số user được đánh giá:", len(eval_users))
print("Số lượt đánh giá phim liên quan:", len(relevant_test))
print("Ngưỡng rating:", RELEVANCE_THRESHOLD)

Số user được đánh giá: 5994
Số lượt đánh giá phim liên quan: 115155
Ngưỡng rating: 4


In [26]:
def evaluate_top_n(
    model,
    eval_users,
    relevant_items_by_user,
    n=10
):
    precisions = []
    recalls = []
    f1_scores = []
    ndcgs = []

    # Phim đã xuất hiện trong Train của mỗi user
    seen_by_user = (
        train_data.groupby("user_id")["movie_id"]
        .apply(set)
        .to_dict()
    )

    for user_id in eval_users:
        u = user_to_idx[user_id]

        # Tính điểm dự đoán cho toàn bộ phim trong tập ứng viên
        scores = (
            model["global_mean"]
            + model["user_bias"][u]
            + model["movie_bias"]
            + model["movie_factors"] @ model["user_factors"][u]
        )

        # Loại bỏ các phim user đã xem trong Train
        seen_movies = seen_by_user.get(user_id, set())
        candidate_mask = ~np.isin(movie_ids, list(seen_movies))

        candidate_movies = movie_ids[candidate_mask]
        candidate_scores = scores[candidate_mask]

        # Chọn Top-N phim có điểm cao nhất
        top_indices = np.argsort(candidate_scores)[-n:][::-1]
        top_movies = candidate_movies[top_indices]

        relevant_items = relevant_items_by_user[user_id]
        hits = [
            1 if movie_id in relevant_items else 0
            for movie_id in top_movies
        ]

        n_hits = sum(hits)

        # Precision@N và Recall@N
        precision = n_hits / n
        recall = n_hits / len(relevant_items)

        # F1@N
        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = 0.0

        # DCG@N
        dcg = sum(
            hit / np.log2(rank + 2)
            for rank, hit in enumerate(hits)
        )

        # IDCG@N: giá trị lý tưởng nếu phim liên quan được xếp đầu
        ideal_hits = min(n, len(relevant_items))
        idcg = sum(
            1 / np.log2(rank + 2)
            for rank in range(ideal_hits)
        )

        ndcg = dcg / idcg if idcg > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        ndcgs.append(ndcg)

    return {
        f"Precision@{n}": np.mean(precisions),
        f"Recall@{n}": np.mean(recalls),
        f"F1@{n}": np.mean(f1_scores),
        f"NDCG@{n}": np.mean(ndcgs),
    }

In [27]:
mf_results = evaluate_top_n(
    model=mf_model,
    eval_users=eval_users,
    relevant_items_by_user=relevant_items_by_user,
    n=10
)

print("Kết quả Matrix Factorization:")
for metric, value in mf_results.items():
    print(f"{metric}: {value:.4f}")

Kết quả Matrix Factorization:
Precision@10: 0.0741
Recall@10: 0.0391
F1@10: 0.0417
NDCG@10: 0.0816


In [28]:
# Tính độ phổ biến chỉ từ dữ liệu Train
popular_movies = (
    train_data.assign(is_relevant=(train_data["rating"] >= 4).astype(int))
    .groupby("movie_id")["is_relevant"]
    .sum()
)

# Đảm bảo thứ tự điểm tương ứng với movie_ids
popularity_scores = (
    pd.Series(movie_ids)
    .map(popular_movies)
    .fillna(0)
    .to_numpy(dtype=np.float32)
)


def evaluate_popularity_top_n(
    eval_users,
    relevant_items_by_user,
    n=10
):
    precisions, recalls, f1_scores, ndcgs = [], [], [], []

    seen_by_user = (
        train_data.groupby("user_id")["movie_id"]
        .apply(set)
        .to_dict()
    )

    for user_id in eval_users:
        seen_movies = seen_by_user.get(user_id, set())

        candidate_mask = ~np.isin(movie_ids, list(seen_movies))
        candidate_movies = movie_ids[candidate_mask]
        candidate_scores = popularity_scores[candidate_mask]

        top_indices = np.argsort(candidate_scores)[-n:][::-1]
        top_movies = candidate_movies[top_indices]

        relevant_items = relevant_items_by_user[user_id]
        hits = [
            int(movie_id in relevant_items)
            for movie_id in top_movies
        ]

        n_hits = sum(hits)
        precision = n_hits / n
        recall = n_hits / len(relevant_items)

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0.0
        )

        dcg = sum(
            hit / np.log2(rank + 2)
            for rank, hit in enumerate(hits)
        )

        ideal_hits = min(n, len(relevant_items))
        idcg = sum(
            1 / np.log2(rank + 2)
            for rank in range(ideal_hits)
        )
        ndcg = dcg / idcg if idcg > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        ndcgs.append(ndcg)

    return {
        f"Precision@{n}": np.mean(precisions),
        f"Recall@{n}": np.mean(recalls),
        f"F1@{n}": np.mean(f1_scores),
        f"NDCG@{n}": np.mean(ndcgs),
    }


baseline_results = evaluate_popularity_top_n(
    eval_users=eval_users,
    relevant_items_by_user=relevant_items_by_user,
    n=10
)

print("Kết quả Popularity Baseline:")
for metric, value in baseline_results.items():
    print(f"{metric}: {value:.4f}")

Kết quả Popularity Baseline:
Precision@10: 0.1519
Recall@10: 0.0905
F1@10: 0.0917
NDCG@10: 0.1766
